# یکسان‌سازی هیستوگرام / Histogram Equalisation

**هدف:** بهبود کنتراست تصاویر با یکسان‌سازی هیستوگرام

---

## محتوا:
1. مفهوم یکسان‌سازی هیستوگرام
2. الگوریتم یکسان‌سازی
3. پیاده‌سازی با OpenCV
4. CLAHE (Contrast Limited Adaptive Histogram Equalization)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV: {cv2.__version__}")

## 1. یکسان‌سازی هیستوگرام چیست؟

**هدف:** توزیع یکنواخت شدت پیکسل‌ها در کل محدوده [0, 255]

**فرمول:**
```
s = T(r) = (L-1) × Σ(i=0 to r) p(i)
```

که در آن:
- `r` = شدت اصلی
- `s` = شدت جدید
- `L` = تعداد سطوح (256)
- `p(i)` = احتمال شدت i

**مراحل:**
1. محاسبه هیستوگرام
2. محاسبه CDF (Cumulative Distribution Function)
3. نرمال‌سازی CDF
4. نگاشت مقادیر جدید

In [ ]:
def histogram_equalisation_manual(image: np.ndarray) -> np.ndarray:
    """
    یکسان‌سازی هیستوگرام به صورت دستی
    """
    # محاسبه هیستوگرام
    hist, _ = np.histogram(image.flatten(), bins=256, range=[0, 256])
    
    # محاسبه CDF
    cdf = hist.cumsum()
    
    # نرمال‌سازی CDF به محدوده [0, 255]
    cdf_normalized = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
    cdf_normalized = cdf_normalized.astype(np.uint8)
    
    # نگاشت مقادیر جدید
    equalised = cdf_normalized[image]
    
    return equalised

print("✓ تابع یکسان‌سازی دستی آماده شد")

## 2. مثال: تصویر تاریک

In [ ]:
# ایجاد تصویر تاریک
dark_image = np.random.randint(0, 80, (256, 256), dtype=np.uint8)
cv2.rectangle(dark_image, (50, 50), (200, 200), 60, -1)
cv2.circle(dark_image, (128, 128), 40, 40, -1)

# یکسان‌سازی
equalised_manual = histogram_equalisation_manual(dark_image)
equalised_opencv = cv2.equalizeHist(dark_image)

# نمایش
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# ردیف اول: تصاویر
axes[0, 0].imshow(dark_image, cmap='gray')
axes[0, 0].set_title('Original (Dark)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(equalised_manual, cmap='gray')
axes[0, 1].set_title('Equalised (Manual)', fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(equalised_opencv, cmap='gray')
axes[0, 2].set_title('Equalised (OpenCV)', fontweight='bold')
axes[0, 2].axis('off')

# ردیف دوم: هیستوگرام‌ها
hist_orig = cv2.calcHist([dark_image], [0], None, [256], [0, 256])
hist_eq_manual = cv2.calcHist([equalised_manual], [0], None, [256], [0, 256])
hist_eq_opencv = cv2.calcHist([equalised_opencv], [0], None, [256], [0, 256])

axes[1, 0].plot(hist_orig, color='black')
axes[1, 0].set_title('Original Histogram')
axes[1, 0].set_xlim([0, 256])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(hist_eq_manual, color='blue')
axes[1, 1].set_title('Equalised Histogram (Manual)')
axes[1, 1].set_xlim([0, 256])
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(hist_eq_opencv, color='red')
axes[1, 2].set_title('Equalised Histogram (OpenCV)')
axes[1, 2].set_xlim([0, 256])
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"تفاوت بین دو روش: {np.sum(np.abs(equalised_manual.astype(int) - equalised_opencv.astype(int)))}")

## 3. نمایش CDF

CDF (Cumulative Distribution Function) نقش کلیدی در یکسان‌سازی دارد.

In [ ]:
def plot_cdf_comparison(image_before: np.ndarray, image_after: np.ndarray):
    """
    مقایسه CDF قبل و بعد از یکسان‌سازی
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # CDF قبل
    hist_before = cv2.calcHist([image_before], [0], None, [256], [0, 256]).flatten()
    cdf_before = hist_before.cumsum()
    cdf_before_normalized = cdf_before / cdf_before.max()
    
    axes[0].plot(cdf_before_normalized, color='blue')
    axes[0].set_title('CDF Before Equalisation', fontweight='bold')
    axes[0].set_xlabel('Intensity')
    axes[0].set_ylabel('Cumulative Probability')
    axes[0].grid(True, alpha=0.3)
    
    # CDF بعد
    hist_after = cv2.calcHist([image_after], [0], None, [256], [0, 256]).flatten()
    cdf_after = hist_after.cumsum()
    cdf_after_normalized = cdf_after / cdf_after.max()
    
    axes[1].plot(cdf_after_normalized, color='red')
    axes[1].set_title('CDF After Equalisation', fontweight='bold')
    axes[1].set_xlabel('Intensity')
    axes[1].set_ylabel('Cumulative Probability')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_cdf_comparison(dark_image, equalised_opencv)

## 4. CLAHE - Contrast Limited Adaptive Histogram Equalization

**مشکل یکسان‌سازی معمولی:** ممکن است نویز را تقویت کند

**راه‌حل CLAHE:**
- تصویر را به بلوک‌های کوچک تقسیم می‌کند
- یکسان‌سازی محلی انجام می‌دهد
- محدودیت روی کنتراست اعمال می‌کند

In [ ]:
# مقایسه Histogram Equalisation و CLAHE

# ایجاد تصویر با نویز
noisy_dark = dark_image.copy()
noise = np.random.normal(0, 15, dark_image.shape).astype(np.int16)
noisy_dark = np.clip(noisy_dark.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# یکسان‌سازی معمولی
eq_normal = cv2.equalizeHist(noisy_dark)

# CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
eq_clahe = clahe.apply(noisy_dark)

# نمایش
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# تصاویر
axes[0, 0].imshow(noisy_dark, cmap='gray')
axes[0, 0].set_title('Original (Noisy)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(eq_normal, cmap='gray')
axes[0, 1].set_title('Standard Equalisation', fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(eq_clahe, cmap='gray')
axes[0, 2].set_title('CLAHE', fontweight='bold')
axes[0, 2].axis('off')

# هیستوگرام‌ها
for idx, (img, title) in enumerate([
    (noisy_dark, 'Original'),
    (eq_normal, 'Standard'),
    (eq_clahe, 'CLAHE')
]):
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    axes[1, idx].plot(hist, color='black')
    axes[1, idx].set_title(f'{title} Histogram')
    axes[1, idx].set_xlim([0, 256])
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. یکسان‌سازی تصاویر رنگی

**نکته:** یکسان‌سازی مستقیم RGB نتیجه خوبی ندارد!

**راه‌حل:** تبدیل به HSV و یکسان‌سازی کانال V

In [ ]:
def equalise_color_image(image_bgr: np.ndarray) -> np.ndarray:
    """
    یکسان‌سازی تصویر رنگی با استفاده از HSV
    """
    # تبدیل به HSV
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    
    # یکسان‌سازی کانال V (Value)
    hsv[:, :, 2] = cv2.equalizeHist(hsv[:, :, 2])
    
    # تبدیل به BGR
    result = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    return result

# ایجاد تصویر رنگی تاریک
color_dark = np.random.randint(0, 80, (256, 256, 3), dtype=np.uint8)
color_dark[50:200, 50:200, 2] = 60  # Red region
color_dark[100:150, 100:150, 1] = 50  # Green region

# یکسان‌سازی
color_equalised = equalise_color_image(color_dark)

# نمایش
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(cv2.cvtColor(color_dark, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Color Image', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(color_equalised, cv2.COLOR_BGR2RGB))
axes[1].set_title('Equalised (HSV method)', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. تمرین عملی

**وظیفه:**
1. یک تصویر واقعی با کنتراست کم بارگذاری کنید
2. هر دو روش (Standard و CLAHE) را اعمال کنید
3. نتایج را مقایسه کنید

In [ ]:
# کد تمرین شما
# TODO: بارگذاری تصویر
# TODO: اعمال یکسان‌سازی
# TODO: مقایسه نتایج

pass

## نتیجه‌گیری

**نکات کلیدی:**

1. **یکسان‌سازی هیستوگرام:** بهبود کنتراست با توزیع یکنواخت
2. **CLAHE:** بهتر از روش معمولی، نویز را کمتر تقویت می‌کند
3. **تصاویر رنگی:** از فضای HSV استفاده کنید
4. **محدودیت:** ممکن است جزئیات را از دست بدهد

---

**بعدی:** `03_histogram_comparison.ipynb` - مقایسه هیستوگرام و back-projection